<h1><center>Laboratory work 6.</center></h1>
<h2><center>PyTorch Going Modular Exercises</center></h2>

**Completed:** Halka Artur

**Variant:** #3

<a class="anchor" id="5"></a>

## Outline

1. [Task 1. Modular Data Preparation](#5.1)
2. [Task 2. Modular Model Creation and Configuration](#5.2)
3. [Task 3. Modular Training and Testing Loops](#5.3)
4. [Task 4. Modular Training Script](#5.4)
5. [Task 5. Modular Prediction Script](#5.5)

In [64]:
# Import torch
import torch
from torch import nn
import os

# Exercises require PyTorch > 1.10.0
print(torch.__version__)

# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("going_modular", exist_ok=True)

2.10.0+cu128


For all tasks below provided, utilize our custom Pizza Steak Sushi (20 percent) dataset from the [GitHub repository](https://github.com/radiukpavlo/conducting-experiments/blob/main/data/pizza_steak_sushi_20_percent.zip).

## <span style="color:red; font-size:1.5em;">Task 1. Modular Data Preparation (`data_setup.py`)</span>

[Go back to the content](#5)

**Variant 3 – Configurable Image Size Pipeline**

- **Goal:** Refactor `create_dataloaders` to support a configurable image size while preserving a fixed square tensor contract for `TinyVGG`.
- **Steps:**
    1. Add an `image_size` argument to `create_dataloaders`, with `64` as the default value.
    2. Construct the resize transform as `transforms.Resize((image_size, image_size))`.
    3. Apply the same deterministic resize and `ToTensor` transform to both training and testing datasets.
    4. Return `class_names` together with both `DataLoader` objects as before.
    5. Inspect one batch for `image_size=64` and another for `image_size=128` to confirm the tensor dimensions change correctly.
- **Hints:** Document that the model classifier may need adjustment if the input image size changes and the architecture is not fully shape-adaptive.

---


In [65]:
%%writefile going_modular/data_setup.py
"""
Contains functionality for downloading image classification data
and creating PyTorch DataLoaders.
"""

import os
import zipfile
import requests
from pathlib import Path

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


DATA_URL = (
    "https://github.com/radiukpavlo/conducting-experiments/raw/main/data/"
    "pizza_steak_sushi_20_percent.zip"
)


def download_data():
    """
    Downloads and extracts dataset if it does not already exist.
    """

    data_path = Path("data")

    train_path = data_path / "train"
    test_path = data_path / "test"

    if train_path.exists() and test_path.exists():
        print("Dataset already exists.")
        return data_path

    print("Downloading dataset...")

    data_path.mkdir(parents=True, exist_ok=True)

    zip_path = data_path / "pizza_steak_sushi_20_percent.zip"

    response = requests.get(DATA_URL)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(data_path)

    os.remove(zip_path)

    print("Dataset downloaded and extracted.")

    return data_path


def create_dataloaders(
    train_dir: str,
    test_dir: str,
    batch_size: int,
    num_workers: int,
    image_size: int = 64
):
    """
    Creates training and testing DataLoaders.
    """

    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor()
    ])

    train_data = datasets.ImageFolder(
        root=train_dir,
        transform=transform
    )

    test_data = datasets.ImageFolder(
        root=test_dir,
        transform=transform
    )

    class_names = train_data.classes

    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_dataloader, test_dataloader, class_names

Overwriting going_modular/data_setup.py


<a class="anchor" id="5.2"></a>

## <span style="color:red; font-size:1.5em;">Task 2. Modular Model Creation and Configuration (`model_builder.py`, `train.py` argparse)</span>

[Go back to the content](#5)


**Variant 3 – Configurable Classifier Shape**

- **Goal:** Make the classifier robust when the image size changes from the default `64x64` setting.
- **Steps:**
    1. Add an `input_image_size` argument to the model constructor or to a model factory function.
    2. Use a dummy tensor during initialization to infer the flattened feature dimension.
    3. Build the final `nn.Linear` layer using the inferred dimension.
    4. Test the model with dummy tensors for `64x64` and `128x128` inputs.
    5. Document that the model and data transform image sizes must match.
- **Hints:** Shape inference reduces hardcoded assumptions but should be implemented inside a clearly named helper.

---

In [66]:
%%writefile going_modular/model_builder.py
"""
Contains PyTorch model code to instantiate a TinyVGG model.
"""
import torch
from torch import nn

class TinyVGG(nn.Module):
    def __init__(
        self,
        input_shape: int,
        hidden_units: int,
        output_shape: int,
        input_image_size: int = 64
    ):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        # for feature size
        dummy_input = torch.randn(
            1,
            input_shape,
            input_image_size,
            input_image_size
        )

        x = self.conv_block_1(dummy_input)
        x = self.conv_block_2(x)

        flattened_features = (
            x.shape[1] *
            x.shape[2] *
            x.shape[3]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                in_features=flattened_features,
                out_features=output_shape
            )
        )

    def forward(self, x: torch.Tensor):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)

        return x

Overwriting going_modular/model_builder.py


In [67]:
%%writefile going_modular/train.py
"""
Trains a PyTorch image classification model using device-agnostic code.
"""

import argparse
import os
import torch


from model_builder import TinyVGG
from data_setup import (
    download_data,
    create_dataloaders
)


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--image_size",
        type=int,
        default=64,
        help="Input image size"
    )

    parser.add_argument(
        "--batch_size",
        type=int,
        default=32,
        help="Batch size"
    )

    parser.add_argument(
        "--hidden_units",
        type=int,
        default=10,
        help="Number of hidden units"
    )

    return parser.parse_args()


def main():
    args = parse_args()

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    image_path = download_data()

    train_dataloader, test_dataloader, class_names = create_dataloaders(
        train_dir=image_path / "train",
        test_dir=image_path / "test",
        batch_size=args.batch_size,
        num_workers=os.cpu_count(),
        image_size=args.image_size
    )

    model = TinyVGG(
        input_shape=3,
        hidden_units=args.hidden_units,
        output_shape=len(class_names),
        input_image_size=args.image_size
    ).to(device)

    dummy_input = torch.randn(
        1,
        3,
        args.image_size,
        args.image_size
    ).to(device)

    output = model(dummy_input)

    print(f"Input size: {args.image_size}x{args.image_size}")
    print(f"Output shape: {output.shape}")
    print(model)


if __name__ == "__main__":
    main()

Overwriting going_modular/train.py


In [63]:
!python going_modular/train.py --image_size 128
!python going_modular/train.py --image_size 64

Dataset already exists.
Input size: 128x128
Output shape: torch.Size([1, 3])
TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=8410, out_features=3, bias=True)
  )
)
Dataset already exists.
Input size: 64x64
Output shape: torch.Size([1, 3])
TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1,

<a class="anchor" id="5.3"></a>

## <span style="color:red; font-size:1.5em;">Task 3. Modular Training and Testing Loops (`engine.py`)</span>

[Go back to the content](#5)



**Variant 3 – Image-Size-Aware Engine Smoke Test**

- **Goal:** Verify that the engine works with the configured image size and does not assume a specific spatial resolution.
- **Steps:**
    1. Create a small dummy `DataLoader` with image tensors matching the configured `image_size`.
    2. Run `train_step` for one dummy batch using a compatible model.
    3. Run `test_step` with `torch.inference_mode()` on the same dummy loader.
    4. Print returned losses and accuracies to confirm the interface.
    5. Repeat the check for another image size if the model supports it.
- **Hints:** The engine should depend on tensor shapes accepted by the model, not on hardcoded image dimensions.

---

In [68]:
%%writefile going_modular/engine.py
"""
Contains training and testing loops for PyTorch models.
"""

from typing import Tuple

import torch
from torch import nn


def accuracy_fn(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc


def train_step(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
) -> Tuple[float, float]:

    model.train()

    train_loss = 0.0
    train_acc = 0.0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # forward pass
        y_pred = model(X)

        # loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # optimizer
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accuracy
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += accuracy_fn(y, y_pred_class)

    train_loss /= len(dataloader)
    train_acc /= len(dataloader)

    return train_loss, train_acc


def test_step(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: nn.Module,
    device: torch.device
) -> Tuple[float, float]:

    model.eval()

    test_loss = 0.0
    test_acc = 0.0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            # forward
            test_pred = model(X)

            # loss
            loss = loss_fn(test_pred, y)
            test_loss += loss.item()

            # accuracy
            test_pred_class = test_pred.argmax(dim=1)
            test_acc += accuracy_fn(y, test_pred_class)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)

    return test_loss, test_acc

Overwriting going_modular/engine.py


In [69]:
from going_modular.engine import train_step, test_step
from going_modular.model_builder import TinyVGG

import torch
from torch import nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

class DummyLoader:
    def __init__(self, batch_size=8, image_size=64, num_classes=3, steps=2):
        self.batch_size = batch_size
        self.image_size = image_size
        self.num_classes = num_classes
        self.steps = steps

    def __iter__(self):
        for _ in range(self.steps):
            X = torch.randn(
                self.batch_size, 3,
                self.image_size,
                self.image_size
            )
            y = torch.randint(0, self.num_classes, (self.batch_size,))
            yield X, y

    def __len__(self):
        return self.steps


def make_dummy_loader(batch_size=8, image_size=64, num_classes=3, steps=2):
    return DummyLoader(batch_size, image_size, num_classes, steps)


# -------------------
image_size = 64
num_classes = 3

loss_fn = nn.CrossEntropyLoss()

# -------------------
print("\n--- Engine Smoke Test (64x64) ---")

model = TinyVGG(
    input_shape=3,
    hidden_units=10,
    output_shape=num_classes,
    input_image_size=image_size
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loss, train_acc = train_step(
    model=model,
    dataloader=make_dummy_loader(image_size=64),
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device
)

print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

test_loss, test_acc = test_step(
    model=model,
    dataloader=make_dummy_loader(image_size=64),
    loss_fn=loss_fn,
    device=device
)

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")


# -------------------
print("\n--- Engine Smoke Test (128x128) ---")

model_128 = TinyVGG(
    input_shape=3,
    hidden_units=10,
    output_shape=num_classes,
    input_image_size=128
).to(device)

optimizer_128 = optim.Adam(model_128.parameters(), lr=0.001)

train_loss, train_acc = train_step(
    model=model_128,
    dataloader=make_dummy_loader(image_size=128),
    loss_fn=loss_fn,
    optimizer=optimizer_128,
    device=device
)

print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

test_loss, test_acc = test_step(
    model=model_128,
    dataloader=make_dummy_loader(image_size=128),
    loss_fn=loss_fn,
    device=device
)

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")


--- Engine Smoke Test (64x64) ---
Train Loss: 1.0981 | Train Acc: 37.50%
Test Loss: 1.1032 | Test Acc: 37.50%

--- Engine Smoke Test (128x128) ---
Train Loss: 0.8964 | Train Acc: 50.00%
Test Loss: 2.0265 | Test Acc: 18.75%


<a class="anchor" id="5.4"></a>

## <span style="color:red; font-size:1.5em;">Task 4. Modular Training Script (`train.py`)</span>

[Go back to the content](#5)



**Variant 3 – Image Size Argument Integration**

- **Goal:** Ensure the script passes a consistent image-size setting to both data preparation and model creation.
- **Steps:**
    1. Add `--image_size` to `argparse`, defaulting to `64`.
    2. Pass the value to `create_dataloaders`.
    3. Pass the same value to the model factory if the model needs it for classifier shape inference.
    4. Print image size in the run configuration.
    5. Run a one-batch smoke test before full training starts.
- **Hints:** A single command-line value should control the image-size contract across the full training pipeline.

---

In [70]:
%%writefile going_modular/train_2.py
"""
Train script with CLI image-size control.
"""

import argparse
import os
import torch
from torch import nn

from data_setup import download_data, create_dataloaders
from model_builder import TinyVGG
from engine import train_step, test_step


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--image_size",
        type=int,
        default=64,
        help="Input image size"
    )

    parser.add_argument(
        "--batch_size",
        type=int,
        default=32,
        help="Batch size"
    )

    return parser.parse_args()


def main():
    args = parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n--- Run Configuration ---")
    print(f"Image Size: {args.image_size}")
    print(f"Batch Size: {args.batch_size}")
    print(f"Device: {device}")

    # data
    image_path = download_data()

    train_dataloader, test_dataloader, class_names = create_dataloaders(
        train_dir=image_path / "train",
        test_dir=image_path / "test",
        batch_size=args.batch_size,
        num_workers=os.cpu_count(),
        image_size=args.image_size
    )

    # model
    model = TinyVGG(
        input_shape=3,
        hidden_units=10,
        output_shape=len(class_names),
        input_image_size=args.image_size
    ).to(device)

    # smoke test
    print("\n--- One-Batch Smoke Test ---")

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    train_loss, train_acc = train_step(
        model=model,
        dataloader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device
    )

    print(f"Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")

    test_loss, test_acc = test_step(
        model=model,
        dataloader=test_dataloader,
        loss_fn=loss_fn,
        device=device
    )

    print(f"Test -> Loss: {test_loss:.4f} | Acc: {test_acc:.2f}%")

    #  checkpoint
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "image_size": args.image_size,
        "class_names": class_names
    }

    os.makedirs("models", exist_ok=True)
    torch.save(
        checkpoint,
        f"models/tinyvgg_img.pth"
    )

    print("\nCheckpoint saved.")


if __name__ == "__main__":
    main()

Overwriting going_modular/train_2.py


In [71]:
!python going_modular/train_2.py --image_size 64 --batch_size 32


--- Run Configuration ---
Image Size: 64
Batch Size: 32
Device: cuda
Dataset already exists.

--- One-Batch Smoke Test ---
Train -> Loss: 1.1049 | Acc: 27.50%
Test -> Loss: 1.1006 | Acc: 28.75%

Checkpoint saved.


<a class="anchor" id="5.5"></a>

## <span style="color:red; font-size:1.5em;">Task 5. Modular Prediction Script (`predict.py`)</span>

[Go back to the content](#5)


**Variant 3 – Configurable Image Size Prediction**

- **Goal:** Ensure prediction preprocessing uses the image size stored during training.
- **Steps:**
    1. Read `image_size` from checkpoint preprocessing metadata.
    2. Build `transforms.Resize((image_size, image_size))`.
    3. Convert the image to RGB before applying the transform.
    4. Add a batch dimension with `unsqueeze(0)`.
    5. Confirm that the tensor shape matches the model's expected input.
- **Hints:** Allow a command-line override only if it is clearly marked as advanced and logged.

---


In [72]:
%%writefile going_modular/predict.py
"""
Prediction script for TinyVGG using a trained checkpoint.
"""

import torch
from PIL import Image
from torchvision import transforms

from model_builder import TinyVGG


def predict(image_path: str, model_path: str, device: torch.device):
    # load checkpoint
    checkpoint = torch.load(model_path, map_location=device)

    image_size = checkpoint["image_size"]
    class_names = checkpoint["class_names"]

    print(f"Loaded image_size from checkpoint: {image_size}")

    # rebuild model
    model = TinyVGG(
        input_shape=3,
        hidden_units=10,
        output_shape=len(class_names),
        input_image_size=image_size
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    #  preprocessing
    preprocess = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor()
    ])

    img = Image.open(image_path).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0)

    print(f"Input shape: {img_tensor.shape}")
    assert img_tensor.shape == (1, 3, image_size, image_size)

    # inference
    with torch.inference_mode():
        logits = model(img_tensor.to(device))
        pred_class = torch.argmax(logits, dim=1).item()

    return class_names[pred_class], logits


def main():
    import argparse

    parser = argparse.ArgumentParser()

    parser.add_argument("--image_path", type=str, required=True)
    parser.add_argument("--model_path", type=str, default="models/tinyvgg_img.pth")

    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    pred_class, logits = predict(
        image_path=args.image_path,
        model_path=args.model_path,
        device=device
    )

    print(f"\nPredicted class: {pred_class}")
    print(f"Logits: {logits}")


if __name__ == "__main__":
    main()

Overwriting going_modular/predict.py


In [77]:

!python going_modular/predict.py \
  --image_path /content/data/test/pizza/2111981.jpg \
  --model_path models/tinyvgg_img.pth

Loaded image_size from checkpoint: 64
Input shape: torch.Size([1, 3, 64, 64])

Predicted class: pizza
Logits: tensor([[ 0.0081, -0.0431, -0.0215]], device='cuda:0')
